# 118. Voronoi分割: nomic-embed-text-v2-moe の評価とセントロイドエクスポート

## 目的
- nomic-embed-text-v2-moe (768D, MoE) でVoronoi分割を評価
- E5-base (768D) / MiniLM (384D) との比較
- セントロイドをNumPy + JSON形式でエクスポート

## モデル特性
| | nomic-v2-moe | multilingual-e5-base | all-MiniLM-L6-v2 |
|---|---|---|---|
| 次元 | 768 | 768 | 384 |
| 言語 | 多言語 | 多言語 | 英語 |
| プレフィックス | search_document:/search_query: | query:/passage: | **不要** |
| アーキテクチャ | MoE | Dense | Dense |

## NB114参考値 (E5-base EN, C=256)
| assign | P | R@10 | 候補% |
|--------|---|------|-------|
| 2 | 2 | 78.9% | 2.8% |
| 2 | 5 | 88.5% | 6.5% |

## 前提
- NB96で保存した `10k_nomic_v2_moe_{ja,en}_embeddings.npy` を使用

## 0. Setup

In [1]:
import numpy as np
import json
from pathlib import Path
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = Path('../data')
np.random.seed(42)

MODEL_KEY = 'nomic_v2_moe'
N_QUERIES = 200
TOP_K = 10

## 1. データロードと空間分析

In [2]:
# NB96で保存したEmbeddingをロード
ja_path = DATA_DIR / f'10k_{MODEL_KEY}_ja_embeddings.npy'
en_path = DATA_DIR / f'10k_{MODEL_KEY}_en_embeddings.npy'

for p in [ja_path, en_path]:
    if not p.exists():
        raise FileNotFoundError(f'{p} が見つかりません。先にNB96を実行してください。')

emb_ja = np.load(ja_path).astype(np.float32)
emb_en = np.load(en_path).astype(np.float32)
print(f'JA: {emb_ja.shape}, EN: {emb_en.shape}')

# 正規化
emb_ja_normed = emb_ja / np.linalg.norm(emb_ja, axis=1, keepdims=True)
emb_en_normed = emb_en / np.linalg.norm(emb_en, axis=1, keepdims=True)

# JA+EN混合
emb_mixed = np.vstack([emb_ja_normed, emb_en_normed])
print(f'Mixed: {emb_mixed.shape}')

# 空間分析（NB113/NB117と同じ）
rng = np.random.default_rng(42)

for lang, emb_normed in [('JA', emb_ja_normed), ('EN', emb_en_normed)]:
    idx1 = rng.choice(len(emb_normed), 5000, replace=True)
    idx2 = rng.choice(len(emb_normed), 5000, replace=True)
    mask = idx1 != idx2
    cos_rand = np.sum(emb_normed[idx1[mask]] * emb_normed[idx2[mask]], axis=1)

    query_ids = rng.choice(len(emb_normed), 200, replace=False)
    knn_sims = []
    for qi in query_ids:
        sims = emb_normed[qi] @ emb_normed.T
        sims[qi] = -1
        knn_sims.append(np.mean(np.sort(sims)[-10:]))

    print(f'\n=== {lang} Embedding空間分析 ===')
    print(f'  ランダムペア cos: mean={cos_rand.mean():.4f}, std={cos_rand.std():.4f}')
    print(f'  k=10近傍 cos:     mean={np.mean(knn_sims):.4f}, std={np.std(knn_sims):.4f}')
    print(f'  Gap:              {np.mean(knn_sims) - cos_rand.mean():.4f}')

print(f'\n--- 参考値 ---')
print(f'  E5-base EN: ランダム=0.706, 近傍=0.815, Gap=0.109')
print(f'  E5-base JA: ランダム=0.765, 近傍=0.873, Gap=0.108')
print(f'  MiniLM EN:  ランダム=0.020, 近傍=0.440, Gap=0.421')

JA: (9990, 768), EN: (10000, 768)
Mixed: (19990, 768)

=== JA Embedding空間分析 ===
  ランダムペア cos: mean=0.2726, std=0.0891
  k=10近傍 cos:     mean=0.6333, std=0.0862
  Gap:              0.3606



=== EN Embedding空間分析 ===
  ランダムペア cos: mean=0.1714, std=0.0741
  k=10近傍 cos:     mean=0.4957, std=0.0792
  Gap:              0.3243

--- 参考値 ---
  E5-base EN: ランダム=0.706, 近傍=0.815, Gap=0.109
  E5-base JA: ランダム=0.765, 近傍=0.873, Gap=0.108
  MiniLM EN:  ランダム=0.020, 近傍=0.440, Gap=0.421


## 2. k-means構築とグリッドサーチ (assign=1,2,3 × C × P)

In [3]:
def precompute_ground_truth(embeddings, query_indices, top_k=10):
    gt_dict = {}
    cos_all = cosine_similarity(embeddings[query_indices], embeddings)
    for i, qi in enumerate(query_indices):
        cos_all[i, qi] = -1
        gt_dict[qi] = set(np.argsort(cos_all[i])[-top_k:])
    return gt_dict


def build_multi_assign(all_sims, n_assign):
    N, C = all_sims.shape
    partitions = {c: [] for c in range(C)}
    for i in range(N):
        for pid in np.argsort(-all_sims[i])[:n_assign]:
            partitions[pid].append(i)
    for c in range(C):
        partitions[c] = np.array(partitions[c], dtype=int)
    return partitions


def evaluate(emb_normed, centroids, partitions, gt_dict, qi, n_probes, emb, top_k=10):
    recalls, cands_list = [], []
    for q in qi:
        gt = gt_dict[q]
        top_c = np.argsort(-(centroids @ emb_normed[q]))[:n_probes]
        cands = set()
        for c in top_c:
            cands.update(partitions[c].tolist())
        cands.discard(q)
        cands = np.array(list(cands))
        cands_list.append(len(cands))
        if len(cands) > 0:
            s = cosine_similarity(emb[q:q+1], emb[cands])[0]
            top_in = cands[np.argsort(-s)[:top_k]]
            recalls.append(len(gt & set(top_in)) / top_k)
        else:
            recalls.append(0.0)
    return np.mean(recalls), np.mean(cands_list)

In [4]:
# 言語別にグリッドサーチ
cluster_range = [32, 64, 128, 256]
probe_range = [1, 2, 3, 4, 5, 8, 10, 15, 20]
assign_range = [1, 2, 3]

all_results = {}

for lang, emb_normed, emb in [
    ('JA', emb_ja_normed, emb_ja),
    ('EN', emb_en_normed, emb_en),
]:
    print(f'\n{"="*70}')
    print(f'{lang}: {emb.shape}')
    print(f'{"="*70}')

    # k-meansモデル構築（混合データで学習）
    kmeans_models = {}
    for n_c in cluster_range:
        km = MiniBatchKMeans(n_clusters=n_c, random_state=42, batch_size=2048, n_init=3)
        km.fit(emb_mixed)
        c = km.cluster_centers_
        c_normed = c / np.linalg.norm(c, axis=1, keepdims=True)
        all_sims = emb_normed @ c_normed.T
        kmeans_models[n_c] = {'centroids': c_normed, 'all_sims': all_sims}
        sizes = [np.sum(km.labels_ == i) for i in range(n_c)]
        print(f'  C={n_c}: mean={np.mean(sizes):.1f}, min={np.min(sizes)}, '
              f'max={np.max(sizes)}, CV={np.std(sizes)/np.mean(sizes):.3f}')

    # GT事前計算
    qi = rng.choice(len(emb), N_QUERIES, replace=False)
    gt = precompute_ground_truth(emb, qi)
    print(f'  {N_QUERIES} queries, GT precomputed')

    # グリッドサーチ
    results = []
    for n_c in cluster_range:
        m = kmeans_models[n_c]
        for n_a in assign_range:
            parts = build_multi_assign(m['all_sims'], n_a)
            for n_p in probe_range:
                if n_p > n_c:
                    continue
                r, c = evaluate(emb_normed, m['centroids'], parts, gt, qi, n_p, emb)
                results.append({
                    'n_clusters': n_c, 'n_assign': n_a, 'n_probes': n_p,
                    'recall': r, 'candidates': c, 'cand_ratio': c / len(emb),
                })

    all_results[lang] = results
    print(f'  Total configs: {len(results)}')

# C=256のセントロイドを保持（エクスポート用）
# 混合データで再学習
km_256 = MiniBatchKMeans(n_clusters=256, random_state=42, batch_size=2048, n_init=3)
km_256.fit(emb_mixed)
centroids_256 = km_256.cluster_centers_
centroids_256_normed = centroids_256 / np.linalg.norm(centroids_256, axis=1, keepdims=True)
print(f'\nCentroids (C=256): {centroids_256_normed.shape}')


JA: (9990, 768)


  C=32: mean=624.7, min=194, max=993, CV=0.369


  C=64: mean=312.3, min=21, max=827, CV=0.495


  C=128: mean=156.2, min=2, max=719, CV=0.609


  C=256: mean=78.1, min=1, max=318, CV=0.809
  200 queries, GT precomputed


...

  C=64: mean=312.3, min=21, max=827, CV=0.495


  C=128: mean=156.2, min=2, max=719, CV=0.609


  C=256: mean=78.1, min=1, max=318, CV=0.809
  200 queries, GT precomputed


  Total configs: 108



Centroids (C=256): (256, 768)


## 3. C=256 結果テーブル と E5-base/MiniLM比較

In [5]:
for lang in ['JA', 'EN']:
    results = all_results[lang]
    print(f'\n{"="*70}')
    print(f'C=256 結果テーブル ({lang})')
    print(f'{"="*70}')

    for n_a in assign_range:
        print(f'\n--- assign={n_a} ---')
        print(f'{"P":>4} {"R@10":>8} {"候補数":>8} {"候補%":>8}')
        print('-' * 32)
        for r in sorted([x for x in results if x['n_clusters']==256 and x['n_assign']==n_a],
                        key=lambda x: x['n_probes']):
            print(f'{r["n_probes"]:>4} {r["recall"]*100:>7.1f}% '
                  f'{r["candidates"]:>7.0f} {r["cand_ratio"]*100:>7.1f}%')

# E5-base EN (NB114) との比較
print(f'\n{"="*70}')
print('E5-base EN (NB114) vs nomic 比較 (C=256, assign=2)')
print(f'{"="*70}')

e5_ref = {
    2: (78.9, 2.8), 3: (83.8, 4.0), 5: (88.5, 6.5),
    8: (92.4, 10.1), 10: (94.2, 12.4),
}

# MiniLM EN (NB117) 参考値
minilm_ref = {
    2: (79.6, 2.0), 3: (84.7, 2.8), 5: (89.6, 4.3),
    8: (93.1, 6.3), 10: (94.5, 7.6),
}

print(f'{"P":>4} {"E5 R@10":>10} {"MiniLM R@10":>12} {"nomic EN R@10":>14} {"nomic JA R@10":>14}')
print('-' * 58)
for n_p in [2, 3, 5, 8, 10]:
    en_r = [x for x in all_results['EN'] if x['n_clusters']==256 and x['n_assign']==2 and x['n_probes']==n_p]
    ja_r = [x for x in all_results['JA'] if x['n_clusters']==256 and x['n_assign']==2 and x['n_probes']==n_p]
    if en_r and n_p in e5_ref:
        e5_r, _ = e5_ref[n_p]
        ml_r, _ = minilm_ref[n_p]
        print(f'{n_p:>4} {e5_r:>9.1f}% {ml_r:>11.1f}% '
              f'{en_r[0]["recall"]*100:>13.1f}% {ja_r[0]["recall"]*100:>13.1f}%')


C=256 結果テーブル (JA)

--- assign=1 ---
   P     R@10      候補数      候補%
--------------------------------
   1    62.5%     113     1.1%
   2    80.2%     220     2.2%
   3    88.2%     327     3.3%
   4    91.5%     428     4.3%
   5    93.3%     523     5.2%
   8    96.4%     769     7.7%
  10    97.2%     924     9.2%
  15    98.5%    1302    13.0%
  20    98.9%    1675    16.8%

--- assign=2 ---
   P     R@10      候補数      候補%
--------------------------------
   1    77.2%     213     2.1%
   2    90.2%     370     3.7%
   3    94.4%     527     5.3%
   4    96.3%     658     6.6%
   5    97.2%     782     7.8%
   8    98.8%    1123    11.2%
  10    99.2%    1331    13.3%
  15    99.4%    1853    18.6%
  20    99.6%    2347    23.5%

--- assign=3 ---
   P     R@10      候補数      候補%
--------------------------------
   1    84.8%     312     3.1%
   2    93.7%     515     5.2%
   3    96.7%     715     7.2%
   4    97.8%     879     8.8%
   5    98.3%    1027    10.3%
   8    99.5%    14

## 4. Recall目標別の推奨構成

In [6]:
for lang in ['EN', 'JA']:
    results = all_results[lang]
    print(f'\n{"="*70}')
    print(f'Recall目標別 最小コスト構成 ({lang})')
    print(f'{"="*70}')

    for target in [0.95, 0.90, 0.85, 0.80, 0.75]:
        candidates = [r for r in results if r['recall'] >= target]
        if not candidates:
            print(f'  R@10>={target*100:.0f}%: 達成する構成なし')
            continue
        best = min(candidates, key=lambda x: x['candidates'])
        config = f'C={best["n_clusters"]},A={best["n_assign"]},P={best["n_probes"]}'
        print(f'  R@10>={target*100:.0f}%: {config:<22} R@10={best["recall"]*100:.1f}% '
              f'候補={best["candidates"]:.0f} ({best["cand_ratio"]*100:.1f}%) IN句={best["n_probes"]}')


Recall目標別 最小コスト構成 (EN)
  R@10>=95%: C=256,A=3,P=5          R@10=95.4% 候補=1012 (10.1%) IN句=5
  R@10>=90%: C=256,A=2,P=4          R@10=91.0% 候補=636 (6.4%) IN句=4
  R@10>=85%: C=256,A=1,P=5          R@10=85.1% 候補=469 (4.7%) IN句=5
  R@10>=80%: C=256,A=2,P=2          R@10=82.6% 候補=362 (3.6%) IN句=2
  R@10>=75%: C=256,A=3,P=1          R@10=78.4% 候補=300 (3.0%) IN句=1

Recall目標別 最小コスト構成 (JA)
  R@10>=95%: C=256,A=2,P=4          R@10=96.3% 候補=658 (6.6%) IN句=4
  R@10>=90%: C=256,A=2,P=2          R@10=90.2% 候補=370 (3.7%) IN句=2
  R@10>=85%: C=256,A=1,P=3          R@10=88.2% 候補=327 (3.3%) IN句=3
  R@10>=80%: C=256,A=1,P=2          R@10=80.2% 候補=220 (2.2%) IN句=2
  R@10>=75%: C=256,A=2,P=1          R@10=77.2% 候補=213 (2.1%) IN句=1


## 5. セントロイドエクスポート (NumPy + JSON)

In [7]:
# NumPy
npy_path = DATA_DIR / f'voronoi_centroids_256_{MODEL_KEY}_mixed.npy'
np.save(npy_path, centroids_256_normed)
loaded = np.load(npy_path)
assert np.allclose(loaded, centroids_256_normed)
print(f'NumPy: {npy_path.name} ({npy_path.stat().st_size/1024:.1f} KB)')
print(f'  Shape: {loaded.shape}, dtype: {loaded.dtype}')

# JSON（推奨構成をresultsから算出）
json_path = DATA_DIR / f'voronoi_centroids_256_{MODEL_KEY}_mixed.json'

rec = {}
# EN結果で推奨構成を決定
en_results = all_results['EN']
for target, label in [(0.90, 'high_recall'), (0.85, 'balanced'), (0.80, 'low_cost')]:
    cands = [r for r in en_results if r['recall'] >= target]
    if cands:
        best = min(cands, key=lambda x: x['candidates'])
        rec[label] = {
            'assign': best['n_assign'], 'probes': best['n_probes'],
            'note': f'R@10>={target*100:.0f}%',
            'recall_en': round(best['recall'] * 100, 1),
            'candidate_ratio': round(best['cand_ratio'] * 100, 1),
        }

export_data = {
    'metadata': {
        'model': 'nomic-ai/nomic-embed-text-v2-moe',
        'dimension': int(centroids_256_normed.shape[1]),
        'n_clusters': int(centroids_256_normed.shape[0]),
        'training_data': 'Wikipedia 10K EN + 10K JA (mixed)',
        'training_samples': int(len(emb_mixed)),
        'normalized': True,
        'prefix': {
            'document': 'search_document: ',
            'query': 'search_query: ',
        },
        'kmeans_params': {
            'random_state': 42,
            'batch_size': 2048,
            'n_init': 3,
        },
        'recommended_configs': rec,
        'usage': {
            'zope': 'KeywordIndex pivot_ids, query with operator="or"',
            'firestore': 'array field pivot_ids, query with array-contains-any',
        },
    },
    'centroids': centroids_256_normed.tolist(),
}

with open(json_path, 'w') as f:
    json.dump(export_data, f, ensure_ascii=False)

print(f'JSON:  {json_path.name} ({json_path.stat().st_size/1024:.1f} KB)')

# 検証
with open(json_path) as f:
    d = json.load(f)
assert np.array(d['centroids']).shape == centroids_256_normed.shape
print(f'\nMetadata:')
for k, v in d['metadata'].items():
    print(f'  {k}: {v}')

NumPy: voronoi_centroids_256_nomic_v2_moe_mixed.npy (768.1 KB)
  Shape: (256, 768), dtype: float32
JSON:  voronoi_centroids_256_nomic_v2_moe_mixed.json (4246.2 KB)



Metadata:
  model: nomic-ai/nomic-embed-text-v2-moe
  dimension: 768
  n_clusters: 256
  training_data: Wikipedia 10K EN + 10K JA (mixed)
  training_samples: 19990
  normalized: True
  prefix: {'document': 'search_document: ', 'query': 'search_query: '}
  kmeans_params: {'random_state': 42, 'batch_size': 2048, 'n_init': 3}
  recommended_configs: {'high_recall': {'assign': 2, 'probes': 4, 'note': 'R@10>=90%', 'recall_en': 91.0, 'candidate_ratio': 6.4}, 'balanced': {'assign': 1, 'probes': 5, 'note': 'R@10>=85%', 'recall_en': 85.1, 'candidate_ratio': 4.7}, 'low_cost': {'assign': 2, 'probes': 2, 'note': 'R@10>=80%', 'recall_en': 82.6, 'candidate_ratio': 3.6}}
  usage: {'zope': 'KeywordIndex pivot_ids, query with operator="or"', 'firestore': 'array field pivot_ids, query with array-contains-any'}


## 6. 評価・考察

In [8]:
print('='*70)
print('E5-base EN vs MiniLM EN vs nomic EN/JA 比較 (C=256, assign=2)')
print('='*70)

print(f'\n{"P":>4} {"E5 R@10":>10} {"MiniLM R@10":>12} {"nomic EN R@10":>14} {"nomic JA R@10":>14}')
print('-' * 58)
for n_p in [2, 3, 4, 5, 8, 10]:
    en_r = [x for x in all_results['EN'] if x['n_clusters']==256 and x['n_assign']==2 and x['n_probes']==n_p]
    ja_r = [x for x in all_results['JA'] if x['n_clusters']==256 and x['n_assign']==2 and x['n_probes']==n_p]
    e5_r = e5_ref.get(n_p, (None, None))[0]
    ml_r = minilm_ref.get(n_p, (None, None))[0]
    e5_str = f'{e5_r:.1f}%' if e5_r else '-'
    ml_str = f'{ml_r:.1f}%' if ml_r else '-'
    en_str = f'{en_r[0]["recall"]*100:.1f}%' if en_r else '-'
    ja_str = f'{ja_r[0]["recall"]*100:.1f}%' if ja_r else '-'
    print(f'{n_p:>4} {e5_str:>10} {ml_str:>12} {en_str:>14} {ja_str:>14}')

print(f'\n--- エクスポートファイル ---')
print(f'| 形式 | ファイル | サイズ |')
print(f'|------|---------|--------|')
print(f'| NumPy | voronoi_centroids_256_{MODEL_KEY}_mixed.npy | {npy_path.stat().st_size/1024:.0f} KB |')
print(f'| JSON | voronoi_centroids_256_{MODEL_KEY}_mixed.json | {json_path.stat().st_size/1024:.0f} KB |')

print(f'\n--- 推奨構成 ---')
for label, cfg in rec.items():
    print(f'  {label}: assign={cfg["assign"]}, probes={cfg["probes"]} '
          f'-> R@10={cfg["recall_en"]:.1f}% (候補{cfg["candidate_ratio"]:.1f}%)')

print('\n' + '='*70)
print('結論:')
print('- nomic-embed-text-v2-moeのVoronoi分割性能を上記テーブルで確認')
print('- E5-baseと同じC=256, assign=2-3のテンプレートが適用可能')
print('- セントロイドを.npy/.json形式でエクスポート済み')
print('- JSONにはprefixフィールドを含む（search_document:/search_query:）')
print('='*70)

E5-base EN vs MiniLM EN vs nomic EN/JA 比較 (C=256, assign=2)

   P    E5 R@10  MiniLM R@10  nomic EN R@10  nomic JA R@10
----------------------------------------------------------
   2      78.9%        79.6%          82.6%          90.2%
   3      83.8%        84.7%          87.7%          94.4%
   4          -            -          91.0%          96.3%
   5      88.5%        89.6%          92.5%          97.2%
   8      92.4%        93.1%          95.1%          98.8%
  10      94.2%        94.5%          96.0%          99.2%

--- エクスポートファイル ---
| 形式 | ファイル | サイズ |
|------|---------|--------|
| NumPy | voronoi_centroids_256_nomic_v2_moe_mixed.npy | 768 KB |
| JSON | voronoi_centroids_256_nomic_v2_moe_mixed.json | 4246 KB |

--- 推奨構成 ---
  high_recall: assign=2, probes=4 -> R@10=91.0% (候補6.4%)
  balanced: assign=1, probes=5 -> R@10=85.1% (候補4.7%)
  low_cost: assign=2, probes=2 -> R@10=82.6% (候補3.6%)

結論:
- nomic-embed-text-v2-moeのVoronoi分割性能を上記テーブルで確認
- E5-baseと同じC=256, assign=2-3のテンプレ

## 評価と考察

### 1. Embedding空間のGapがVoronoi分割に最適

| モデル | ランダムcos | 近傍cos | Gap | E5比 |
|--------|-----------|---------|-----|------|
| **nomic JA** | 0.273 | 0.633 | **0.361** | 3.3倍 |
| **nomic EN** | 0.171 | 0.496 | **0.324** | 3.0倍 |
| E5-base EN | 0.706 | 0.815 | 0.109 | 1.0倍 |
| E5-base JA | 0.765 | 0.873 | 0.108 | 1.0倍 |
| MiniLM EN | 0.020 | 0.440 | 0.421 | 3.9倍 |

NB96で確認した等方性の高さ（cos_mean=0.17-0.27）がVoronoi分割でも好影響を与え、近傍と非近傍の分離（Gap）がE5-baseの約3倍。k-meansクラスタが意味のある空間分割として機能しやすい。

### 2. 全probe数でE5-base・MiniLMを上回るRecall

C=256, assign=2での比較:

| P | E5 EN | MiniLM EN | nomic EN | nomic JA |
|---|-------|-----------|----------|----------|
| 2 | 78.9% | 79.6% | **82.6%** | **90.2%** |
| 3 | 83.8% | 84.7% | **87.7%** | **94.4%** |
| 5 | 88.5% | 89.6% | **92.5%** | **97.2%** |
| 8 | 92.4% | 93.1% | **95.1%** | **98.8%** |
| 10 | 94.2% | 94.5% | **96.0%** | **99.2%** |

- ENでE5-baseに対して+2〜4pp、MiniLMに対して+1〜3pp
- JAは特に優秀で、P=2でE5 ENのP=5相当（90.2% vs 88.5%）、P=5で99%近い
- 同じRecallをより少ないprobeで達成 → IN句の要素数を減らせる → クエリコスト削減

### 3. JA > ENの性能差の理由

JA（R@10=90.2% at P=2）がEN（82.6%）を大きく上回る。原因はGapの差（JA 0.361 > EN 0.324）。JA Wikipediaテキストはトピックの多様性が高く、クラスタ分離が良好なためと考えられる。

### 4. 推奨構成

| 目標 | 構成 | EN R@10 | JA R@10 | 候補% | IN句 |
|------|------|---------|---------|-------|------|
| R@10>=95% | C=256, A=3, P=5 | 95.4% | 98.3% | 10.1% | 5 |
| R@10>=90% | C=256, A=2, P=4 | 91.0% | 96.3% | 6.4% | 4 |
| R@10>=85% | C=256, A=1, P=5 | 85.1% | 93.3% | 4.7% | 5 |
| R@10>=80% | C=256, A=2, P=2 | 82.6% | 90.2% | 3.6% | 2 |

E5-baseと同じC=256テンプレートがそのまま適用可能。assign=2, P=4が汎用的なバランス構成。

### 5. NB96 ITQ-LSH結果との整合性

NB96でnomicのITQ-LSH Spearmanが全モデル中最高（-0.627）だったことと、Voronoi分割でも最高性能を示したことは一貫している。**等方的なEmbedding空間はハッシュベース・パーティションベースの両方の近似手法に有利**であることを実証。

### 6. 結論

- nomic-embed-text-v2-moeは**Voronoi分割において全評価済みモデル中最高のRecall**を達成
- 等方的な空間構造により、少ないprobe数で高Recallが可能（クエリコスト効率が良い）
- C=256, assign=2-3のテンプレートが有効で、既存インフラ（Firestore/Zope）にそのまま適用可能
- セントロイドをエクスポート済み（prefixメタデータ付きJSON）